# Aurora Siger — Sistema de Análise de Telemetria

Sistema de verificação pré-lançamento com análise por IA (Google Gemini).

> **Pré-requisito:** Adicione sua chave em **Secrets** (ícone de chave no painel esquerdo) com o nome `GEMINI_API_KEY` antes de executar.

## 1. Instalação

Instala o SDK oficial do Gemini (`google-genai`) e a biblioteca de widgets interativos.

In [ ]:
!pip install -q google-genai ipywidgets

## 2. Configuração do cliente Gemini

Lê a chave de API diretamente dos **Secrets** do Colab e inicializa o cliente.

In [ ]:
import os
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
cliente = genai.Client(api_key=GEMINI_API_KEY)
print('✅ Cliente Gemini configurado.')

## 3. Dados de telemetria

Ajuste os sliders com os valores reais da missão. O botão **Integridade estrutural** indica se os módulos físicos estão operacionais.

| Parâmetro | Faixa válida |
|---|---|
| Temperatura interna | -10°C a 40°C |
| Temperatura externa | -5°C a 45°C |
| Gradiente térmico (calculado) | ≤ 40°C |
| Energia | ≥ 70% |
| Combustível RP-1 | ≥ 80% |
| Oxidante LOX | ≥ 80% |
| Pressão do tanque | 2,5 a 4,5 bar |

In [ ]:
# Célula 3 — Entrada de dados via widgets
import ipywidgets as widgets
from IPython.display import display

style  = {'description_width': '260px'}
layout = widgets.Layout(width='500px')

w_temp_interna      = widgets.FloatSlider(value=22.0, min=-10, max=40,  step=0.5, description='Temperatura interna (C):', style=style, layout=layout)
w_temp_externa      = widgets.FloatSlider(value=20.0, min=-5,  max=45,  step=0.5, description='Temperatura externa (C):', style=style, layout=layout)
w_nivel_energia     = widgets.FloatSlider(value=85.0, min=0,   max=100, step=1.0, description='Nivel de energia (%):', style=style, layout=layout)
w_nivel_combustivel = widgets.FloatSlider(value=90.0, min=0,   max=100, step=1.0, description='Combustivel RP-1 (%):', style=style, layout=layout)
w_nivel_oxidante    = widgets.FloatSlider(value=90.0, min=0,   max=100, step=1.0, description='Oxidante LOX (%):', style=style, layout=layout)
w_pressao_tanque    = widgets.FloatSlider(value=3.5,  min=0,   max=6,   step=0.1, description='Pressao do tanque (bar):', style=style, layout=layout)
w_integridade       = widgets.ToggleButtons(value='OK', options=['OK', 'ERRO'], description='Integridade estrutural:', style=style)

print('=' * 50)
print('  AURORA SIGER — Dados de Telemetria')
print('=' * 50)
display(w_temp_interna, w_temp_externa, w_nivel_energia,
        w_nivel_combustivel, w_nivel_oxidante, w_pressao_tanque, w_integridade)


## 4. Verificação e análise IA

Valida cada parâmetro contra os limites operacionais e envia a telemetria ao Gemini para análise de risco. Execute **após** ajustar os sliders acima.

In [ ]:
import time

# INPUTS (lidos dos widgets)
temp_interna      = w_temp_interna.value
temp_externa      = w_temp_externa.value
nivel_energia     = w_nivel_energia.value
nivel_combustivel = w_nivel_combustivel.value
nivel_oxidante    = w_nivel_oxidante.value
pressao_tanque    = w_pressao_tanque.value
integridade       = 1 if w_integridade.value == 'OK' else 0

# CÁLCULOS
gradiente_termico = abs(temp_interna - temp_externa)

print('\n[ Analisando sistema... ]\n')

# VERIFICAÇÕES
resultados = []

if not (-10 <= temp_interna <= 40):
    resultados.append(('Temperatura interna', False, f'Fora do limite ({temp_interna}°C). Esperado: -10°C a 40°C.'))
else:
    resultados.append(('Temperatura interna', True, None))

if not (-5 <= temp_externa <= 45):
    resultados.append(('Temperatura externa', False, f'Fora do limite ({temp_externa}°C). Esperado: -5°C a 45°C.'))
else:
    resultados.append(('Temperatura externa', True, None))

if not (gradiente_termico <= 40):
    resultados.append(('Gradiente térmico', False, f'Gradiente excessivo ({gradiente_termico:.1f}°C). Máx: 40°C.'))
else:
    resultados.append(('Gradiente térmico', True, None))

if not (nivel_energia >= 70):
    resultados.append(('Nível de energia', False, f'Energia insuficiente ({nivel_energia:.1f}%). Mín: 70%.'))
else:
    resultados.append(('Nível de energia', True, None))

if not (nivel_combustivel >= 80):
    resultados.append(('Nível de combustível (RP-1)', False, f'Combustível insuficiente ({nivel_combustivel:.1f}%). Mín: 80%.'))
else:
    resultados.append(('Nível de combustível (RP-1)', True, None))

if not (nivel_oxidante >= 80):
    resultados.append(('Nível de oxidante (LOX)', False, f'Oxidante insuficiente ({nivel_oxidante:.1f}%). Mín: 80%.'))
else:
    resultados.append(('Nível de oxidante (LOX)', True, None))

if not (2.5 <= pressao_tanque <= 4.5):
    resultados.append(('Pressão do tanque', False, f'Pressão fora do intervalo ({pressao_tanque:.2f} bar). Esperado: 2.5–4.5 bar.'))
else:
    resultados.append(('Pressão do tanque', True, None))

if not (integridade == 1):
    resultados.append(('Integridade estrutural', False, 'Falha estrutural detectada.'))
else:
    resultados.append(('Integridade estrutural', True, None))

status_modulos = all(passou for _, passou, _ in resultados)
resultados.append(('Status dos módulos', status_modulos, 'Módulos com falha detectada.'))

for nome, passou, erro in resultados:
    if not passou:
        print(f'  >> {nome}: FALHA')
        print(f'     {erro}')
        print('\n❌ DECOLAGEM ABORTADA')
        break
    print(f'  >> {nome}: OK')
    time.sleep(0.3)
else:
    print('\n✅ PRONTO PARA DECOLAR')

# ANÁLISE IA
status_texto = 'PRONTO PARA DECOLAR' if status_modulos else 'DECOLAGEM ABORTADA'

prompt = f"""
Você é um sistema de análise de telemetria aeroespacial. Analise os dados da Missão Aurora Siger:

TELEMETRIA:
- Temperatura interna: {temp_interna}°C
- Temperatura externa: {temp_externa}°C
- Gradiente térmico: {gradiente_termico:.1f}°C
- Nível de energia: {nivel_energia:.1f}%
- Nível de combustível (RP-1): {nivel_combustivel:.1f}%
- Nível de oxidante (LOX): {nivel_oxidante:.1f}%
- Pressão do tanque: {pressao_tanque:.2f} bar
- Integridade estrutural: {'OK' if integridade == 1 else 'ERRO'}
- Status geral: {status_texto}

Responda em português com:
1. Classificação de cada dado (normal / atenção / crítico)
2. Possíveis anomalias identificadas
3. Sugestões de risco
4. Formato de resposta: Texto simples, direto e claro. Sem formatação adicional.
"""

print('\n[ Análise IA - Gemini ]')
print('\n[ Aguardando resultado... ]\n')
resposta = cliente.models.generate_content(model='gemini-2.5-flash', contents=prompt)
print(resposta.text)